In [138]:
import pandas as pd
import glob
import os

# Step 1: Read manifest as pandas dataframe

In [215]:
bed_file_dict = {
    "Arthrogryposis": "adaptive_regions/annotated/merged/Arthro_2582_padded_top11_manual_annotated.sorted.merged.bed",
    "Epilepsy": "adaptive_regions/annotated/merged/Epilepsy_V2_1662_padded_top8_manual_annotated.sorted.merged.bed",
    "Microcephaly": "adaptive_regions/annotated/merged/Microcephaly_1769_padded_top3_manual_annotated.sorted.merged.bed",
    "Neurodegeneration": "adaptive_regions/annotated/merged/Neurodegen_V2_1704_padded_top8_manual_annotated.sorted.merged.bed",
    "CMRG": "adaptive_regions/annotated/merged/GRCh38_CMRG_benchmark_gene_coordinates_padding10K.bed",
    "WGS": None
}
tr_bed_file_dict = {
    "Arthrogryposis": "analysis/tr_regions/adotto_catalog.hg38.lite.Arthro.bed",
    "Epilepsy": "analysis/tr_regions/adotto_catalog.hg38.lite.Epilepsy.bed",
    "Microcephaly": "analysis/tr_regions/adotto_catalog.hg38.lite.Microcephaly.bed",
    "Neurodegeneration": "analysis/tr_regions/adotto_catalog.hg38.lite.Neurodegen.bed",
    "CMRG": "analysis/tr_regions/adotto_catalog.hg38.lite.CMRG.bed",
    "WGS": 'adotto_catalog.hg38.lite.bed'
}

In [150]:
job_name = 'basecalling_dorado'
threads = 25
partition = 'gpu'
task_num = len(manifest)
task_num_same_time = 8
slurm_folder = os.path.abspath("./slurm_results")
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        pod5_path = row.path_to_flowcell
        output_path = os.path.join(output_folder, row.sample_id)
        cmd = (
            f"commands[{index}]=\"dorado basecaller {model} {pod5_path} -r "
            f"-x cuda:all  --kit-name {kit_name} "
            f"-o {output_path} \"\n"
        )
        sbatch_file.write(cmd)
    sbatch_file.write("\n${commands[$SLURM_ARRAY_TASK_ID]}\n")

In [156]:
job_name = 'fastq_extract'
threads = 10
partition = 'gpu'
task_num = len(manifest)
task_num_same_time = 5
slurm_folder = os.path.abspath("./slurm_results")
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        input_bam = glob.glob(f"{os.path.join(output_folder, row.sample_id)}/calls*.bam")[0]
        output_fastq = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.fastq")}"
        cmd = (
            f"commands[{index}]=\"samtools fastq -@ {threads} "
            f"-T RG,Mm,Ml,MM,ML   "
            f"-0 {output_fastq} {input_bam} \"\n"
        )
        sbatch_file.write(cmd)
    sbatch_file.write("\n${commands[$SLURM_ARRAY_TASK_ID]}\n")

In [158]:
job_name = 'minimap2'
threads = 20
partition = 'gpu'
task_num = len(manifest)
task_num_same_time = 9
slurm_folder = os.path.abspath("./slurm_results")
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        input_fastq = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.fastq")}"
        output_sam = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.sam")}"
        cmd = (
            f"commands[{index}]=\"minimap2 -t {threads} "
            f"-a -Y -y -x map-ont -o {output_sam} "
            f"{reference} {input_fastq} \"\n"
        )
        sbatch_file.write(cmd)
    sbatch_file.write("\n${commands[$SLURM_ARRAY_TASK_ID]}\n")

In [193]:
import pandas as pd
import re, pysam, os

def extract_barcodes(sample_id, barcode_prefix, kit_type):
    if '-' in str(sample_id):
        # Range like 43-45
        start, end = map(int, re.match(r"(\d+)-(\d+)", sample_id).groups())
        return [f"{barcode_prefix}_{kit_type}_barcode{num:02d}" for num in range(start, end + 1)] + [barcode_prefix], [f"barcode{num:02d}" for num in range(start, end + 1)]  + ["unassigned"]
    else:
        start, end = map(int, re.match(r"(\d+)_(\d+)", sample_id).groups())
        return [f"{barcode_prefix}_{kit_type}_barcode{num:02d}" for num in range(start, end + 1)] + [barcode_prefix], [f"barcode{num:02d}" for num in range(start, end + 1)] + ["unassigned"]


def get_readgroup_prefix_and_full_rg(bam_or_sam_path):
    file_ext = os.path.splitext(bam_or_sam_path)[1].lower()
    mode = "rb" if file_ext == ".bam" else "r"

    with pysam.AlignmentFile(bam_or_sam_path, mode, check_sq=False) as aln:
        for read in aln.fetch(until_eof=True):
            if read.has_tag("RG"):
                full_rg = read.get_tag("RG")
                prefix = re.sub(r'_SQK-NBD114-96_barcode\d{2}$', '', full_rg)
                return prefix
        return None

In [285]:
get_readgroup_prefix_and_full_rg("GREGoR_adaptive_sampling/10_12_Gregor_Trio/calls_2025-07-25_T21-05-42.bam")

'c0f950c7-cec4-41d3-9c2f-6e9cd053f776_dna_r10.4.1_e8.2_400bps_sup@v5.2.0'

In [199]:
job_name = 'bam_split'
threads = 5
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 15
slurm_folder = os.path.abspath("./slurm_results")
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        input_sam = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.sam")}"
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_sam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        # print(row.sample_id, rg_list, barcode_list)
        for index_inner, rg in enumerate(rg_list[:3]):
            output_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.bam")
            cmd = (
                f"commands[{index * 3 + index_inner}]=\"samtools view -b@ {threads} "
                f"{input_sam} -r {rg}  "
                f"-o {output_bam_barcoded} \"\n"
            )
            sbatch_file.write(cmd)
    sbatch_file.write("\n${commands[$SLURM_ARRAY_TASK_ID]}\n")

In [200]:
job_name = 'bam_sort'
threads = 5
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 15
slurm_folder = os.path.abspath("./slurm_results")
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        input_sam = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.sam")}"
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_sam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        # print(row.sample_id, rg_list, barcode_list)
        for index_inner, rg in enumerate(rg_list[:3]):
            output_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.bam")
            output_bam_barcoded_sorted = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.sorted.bam")
            cmd = (
                f"commands[{index * 3 + index_inner}]=\"samtools sort -@ {threads} "
                f"-o {output_bam_barcoded_sorted} --write-index  "
                f"{output_bam_barcoded} \"\n"
            )
            sbatch_file.write(cmd)
    sbatch_file.write("\n${commands[$SLURM_ARRAY_TASK_ID]}\n")


In [287]:
job_name = 'sniffles_global'
threads = 10
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 15
slurm_folder = os.path.abspath("./slurm_results")
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
clair3_model = "clair3/bin/models/r1041_e82_400bps_sup_v520"
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        input_bam = glob.glob(f"{os.path.join(output_folder, row.sample_id, f"calls*.bam")}")[0]
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_bam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        for index_inner, rg in enumerate(rg_list[:3]):
            sniffles_output_vcf = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_global.germline.vcf.gz")
            sniffles_output_snf = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_global.germline.snf")
            input_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.sorted.bam")
            cmd = (
                f"commands[{index * 3 + index_inner}]=\"sniffles -i {input_bam_barcoded} "
                f"--reference {reference} -v {sniffles_output_vcf} --allow-overwrite --snf {sniffles_output_snf} "
                f"--sample-id {rg} -t {threads}  --output-rnames   \"\n"
            )
            sbatch_file.write(cmd)
    sbatch_file.write("\n${commands[$SLURM_ARRAY_TASK_ID]}\n")

In [204]:
job_name = 'bam_mosdepth'
threads = 10
partition = 'gpu'
task_num = len(manifest) * 4
task_num_same_time = 15
slurm_folder = os.path.abspath("./slurm_results")
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        input_sam = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.sam")}"
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_sam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        for index_inner, rg in enumerate(rg_list):
            input_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.sorted.bam")

            mosdepth_output_folder = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_mosdepth")
            os.makedirs(mosdepth_output_folder, exist_ok=True)
            prefix = os.path.join(mosdepth_output_folder, f"{barcode_list[index_inner]}")
            if bed_file_loc == None:
                cmd = (
                f"commands[{index * 4 + index_inner}]=\"mosdepth -t {threads} "
                f" -x  "
                f" {prefix} {input_bam} \"\n"
                )
                sbatch_file.write(cmd)
            else:
                cmd = (
                    f"commands[{index * 4 + index_inner}]=\"mosdepth -t {threads} "
                    f"-b {bed_file_loc} -x  "
                    f"{prefix} {input_bam_barcoded} \"\n"
                )
                sbatch_file.write(cmd)
    sbatch_file.write("\n${commands[$SLURM_ARRAY_TASK_ID]}\n")

In [207]:
job_name = 'clair3_local'
threads = 10
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 10
slurm_folder = os.path.abspath("./slurm_results")
envrionment_name =  'clair3'
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
    f"micromamba activate {envrionment_name}\n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
clair3_model = "clair3/bin/models/r1041_e82_400bps_sup_v520"
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]

        input_sam = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.sam")}"
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_sam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        for index_inner, rg in enumerate(rg_list[:3]):
            clair3_output_folder = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_clair3_local")
            input_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.sorted.bam")
            if bed_file_loc == None:
                cmd = (
                    f"commands[{index * 3 + index_inner}]=\"run_clair3.sh -b {input_bam_barcoded} "
                    f"-f {reference} -m {clair3_model} --sample_name={rg} "
                    f"-t {threads} -p ont -o {clair3_output_folder} \"\n"
                )
                
            else:
                cmd = (
                    f"commands[{index * 3 + index_inner}]=\"run_clair3.sh -b {input_bam_barcoded} "
                    f"-f {reference} -m {clair3_model} --bed_fn={bed_file_loc} --sample_name={rg} "
                    f"-t {threads} -p ont -o {clair3_output_folder} \"\n"
                )
            sbatch_file.write(cmd)
    sbatch_file.write("\n${commands[$SLURM_ARRAY_TASK_ID]}\n")

In [266]:
job_name = 'kanpig_pileup'
threads = 4
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 10
slurm_folder = os.path.abspath("./slurm_results")
envrionment_name =  'clair3'
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
    f"micromamba activate {envrionment_name}\n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
clair3_model = "clair3/bin/models/r1041_e82_400bps_sup_v520"
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]

        input_sam = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.sam")}"
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_sam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        for index_inner, rg in enumerate(rg_list[:3]):
            kanpig_output = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_kanpig.plup.gz")
            input_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.sorted.bam")
            cmd = (
                f"commands[{index * 3 + index_inner}]=\"kanpig plup -b {input_bam_barcoded} "
                f"-t {threads} | bedtools sort -header | bgzip > {kanpig_output}   \"\n"
            )
            sbatch_file.write(cmd)
    sbatch_file.write("\neval \"${commands[$SLURM_ARRAY_TASK_ID]}\"\n")
    


In [ ]:
job_name = 'kanpig_gt'
threads = 4
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 15
slurm_folder = os.path.abspath("./slurm_results")
# envrionment_name =  'clair3'
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
    f"micromamba activate {envrionment_name}\n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
clair3_model = "clair3/bin/models/r1041_e82_400bps_sup_v520"
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        input_sam = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.sam")}"
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_sam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        for index_inner, rg in enumerate(rg_list[:3]):
            kanpig_input = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_kanpig.plup.gz")
            input_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.sorted.bam")
            sniffles_output_vcf = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_global.germline.vcf.gz")
            kanpig_output_vcf = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_global.germline.kanpig.genotyped.vcf.gz")

            cmd = (
                f"commands[{index * 3 + index_inner}]=\"kanpig gt --bed {bed_file_loc} --input {sniffles_output_vcf} "
                f"-t {threads} --reads {kanpig_input} --sample {rg} --reference {reference} | bcftools sort -Oz -W -o {kanpig_output_vcf} \"\n"
            )
            sbatch_file.write(cmd)
    sbatch_file.write("\neval \"${commands[$SLURM_ARRAY_TASK_ID]}\"\n")
    


In [335]:
job_name = 'kanpig_trio'
threads = 4
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 15
slurm_folder = os.path.abspath("./slurm_results")
# envrionment_name =  'clair3'
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
    f"micromamba activate {envrionment_name}\n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
clair3_model = "clair3/bin/models/r1041_e82_400bps_sup_v520"
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        input_bam = glob.glob(f"{os.path.join(output_folder, row.sample_id, f"calls*.bam")}")[0]
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_bam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        
        proband_kanpig_input = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[0]}_kanpig.plup.gz")
        mother_kanpig_input = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[1]}_kanpig.plup.gz")
        father_kanpig_input = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[2]}_kanpig.plup.gz")
        proband_input_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[0]}.sorted.bam")
        mother_input_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[1]}.sorted.bam")
        father_input_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[2]}.sorted.bam")
        proband_sniffles_output_vcf = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[0]}_global.germline.vcf.gz")
        mother_sniffles_output_vcf = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[1]}_global.germline.vcf.gz")
        father_sniffles_output_vcf = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[2]}_global.germline.vcf.gz")
        kanpig_output_vcf = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[0]}_global.germline.kanpig.trio.genotyped.vcf.gz")
        cmd = (
            f"commands[{index}]=\"kanpig trio --bed {bed_file_loc} --input {proband_sniffles_output_vcf} --proband {proband_kanpig_input} --mother {mother_kanpig_input} --father {father_kanpig_input} "
            f"-t {threads} --reference {reference} | bcftools sort -Oz -W -o {kanpig_output_vcf} \"\n"
        )  
        sbatch_file.write(cmd)
    sbatch_file.write("\neval \"${commands[$SLURM_ARRAY_TASK_ID]}\"\n")
    


In [250]:
job_name = 'medaka_local'
threads = 10
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 10
slurm_folder = os.path.abspath("./slurm_results")
envrionment_name =  'medaka'
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
    f"eval \"$(micromamba shell hook -s bash)\" \n"
    f"micromamba activate {envrionment_name}\n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
clair3_model = "clair3/bin/models/r1041_e82_400bps_sup_v520"
# analysis_folder = "GREGoR_adaptive_sampling/"
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        tr_bed_file_loc = tr_bed_file_dict[row.bed_file]
        input_sam = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.sam")}"
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_sam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        sample_sex = (row.proband_gender).lower()
        for index_inner, rg in enumerate(rg_list[:3]):
            input_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.HP.bam")
            output_tr = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_tr")
            cmd = (
                f"commands[{index * 3 + index_inner}]=\"medaka tandem --workers {threads} "
                f"--ignore_read_groups --model dna_r10.4.1_e8.2_400bps_sup@v5.2.0:consensus --phasing prephased --sample_name {rg} "
                f"{input_bam_barcoded} {reference} {tr_bed_file_loc} {sample_sex} {output_tr} \"\n"
            ) 
            sbatch_file.write(cmd)
    sbatch_file.write("\neval \"${commands[$SLURM_ARRAY_TASK_ID]}\"\n")

In [336]:
job_name = 'medaka_patho'
threads = 10
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 10
slurm_folder = os.path.abspath("./slurm_results")
envrionment_name =  'medaka'
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
    f"eval \"$(micromamba shell hook -s bash)\" \n"
    f"micromamba activate {envrionment_name}\n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
clair3_model = "clair3/bin/models/r1041_e82_400bps_sup_v520"
# analysis_folder = "GREGoR_adaptive_sampling/"
adotto_pheno = "adaptive_regions/patho_tr/adotto_pheno.bed"
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        # tr_bed_file_loc = tr_bed_file_dict[row.bed_file]
        input_bam = glob.glob(f"{os.path.join(output_folder, row.sample_id, f"calls*.bam")}")[0]
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_bam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        sample_sex = (row.proband_gender).lower()
        for index_inner, rg in enumerate(rg_list[:3]):
            input_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.HP.bam")
            output_tr = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_pheno_tr")
            cmd = (
                f"commands[{index * 3 + index_inner}]=\"medaka tandem --workers {threads} "
                f"--ignore_read_groups --model dna_r10.4.1_e8.2_400bps_sup@v5.2.0:consensus --phasing prephased --sample_name {rg} "
                f"{input_bam_barcoded} {reference} {adotto_pheno} {sample_sex} {output_tr} \"\n"
            ) 
            sbatch_file.write(cmd)
    sbatch_file.write("\neval \"${commands[$SLURM_ARRAY_TASK_ID]}\"\n")

In [213]:
job_name = 'whatshap_single_sample_local_phasing'
threads = 1
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 30
slurm_folder = os.path.abspath("./slurm_results")
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
clair3_model = "clair3/bin/models/r1041_e82_400bps_sup_v520"
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        input_sam = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.sam")}"
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_sam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        for index_inner, rg in enumerate(rg_list[:3]):
            sample_id = f"{row.sample_id}_{barcode_list[index_inner]}"
            clair3_output_folder = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_clair3_local") 
            output_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.sorted.bam")
            vcf_in = f"{clair3_output_folder}/merge_output.vcf.gz"
            vcf_out = os.path.join(output_folder, row.sample_id, f"{sample_id}.local.phased.vcf.gz")
            cmd = f'commands[{index * 3 + index_inner}]="whatshap phase --reference {reference} -o {vcf_out} --ignore-read-groups {vcf_in} {output_bam_barcoded}  "\n'
            sbatch_file.write(cmd)
    sbatch_file.write("\n${commands[$SLURM_ARRAY_TASK_ID]}\n")

In [249]:
job_name = 'whatshap_haplotag'
threads = 3
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 10
slurm_folder = os.path.abspath("./slurm_results")
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
clair3_model = "clair3/bin/models/r1041_e82_400bps_sup_v520"
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        input_sam = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.sam")}"
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_sam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        for index_inner, rg in enumerate(rg_list[:3]):
            sample_id = f"{row.sample_id}_{barcode_list[index_inner]}"
            vcf_out = os.path.join(output_folder, row.sample_id, f"{sample_id}.local.phased.vcf.gz")
            input_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.sorted.bam")
            output_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.HP.bam")
            output_hp_assignment = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.read.HP.gz")
            cmd = f'commands[{index * 3 + index_inner}]="whatshap haplotag --ignore-read-groups --output-haplotag-list {output_hp_assignment} --reference {reference}  --output-threads {threads} -o {output_bam_barcoded} {vcf_out} {input_bam_barcoded} "\n'
            sbatch_file.write(cmd)
    sbatch_file.write("\n${commands[$SLURM_ARRAY_TASK_ID]}\n")

In [269]:
job_name = 'modkit'
threads = 10
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 7
slurm_folder = os.path.abspath("./slurm_results")
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
clair3_model = "clair3/bin/models/r1041_e82_400bps_sup_v520"
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        input_sam = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.sam")}"
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_sam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        for index_inner, rg in enumerate(rg_list[:3]):
            modkit_output_directory = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_modkit")
            input_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.HP.bam")
            cmd = (
                f"commands[{index * 3 + index_inner}]=\"modkit pileup --preset traditional "
                f"--ref {reference} --partition-tag HP --prefix {row.sample_id}_{barcode_list[index_inner]}.HP "
                f"-t {threads} {input_bam_barcoded} {modkit_output_directory}  \"\n"
            )
            sbatch_file.write(cmd)
    sbatch_file.write("\n${commands[$SLURM_ARRAY_TASK_ID]}\n")

In [ ]:
job_name = 'modkit_nohp'
threads = 10
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 7
slurm_folder = os.path.abspath("./slurm_results")
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
clair3_model = "clair3/bin/models/r1041_e82_400bps_sup_v520"
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        input_sam = f"{os.path.join(output_folder, row.sample_id, f"{row.sample_id}.sam")}"
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_sam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        for index_inner, rg in enumerate(rg_list[:3]):
            modkit_output_directory = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_modkit")
            input_bam_barcoded = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}.HP.bam")
            cmd = (
                f"commands[{index * 3 + index_inner}]=\"modkit pileup --preset traditional "
                f"--ref {reference} "
                f"-t {threads} {input_bam_barcoded} {modkit_output_directory}/{row.sample_id}_{barcode_list[index_inner]}.bed  \"\n"
            )
            sbatch_file.write(cmd)
    sbatch_file.write("\n${commands[$SLURM_ARRAY_TASK_ID]}\n")

In [294]:
job_name = 'tdb'
threads = 1
partition = 'gpu'
task_num = len(manifest) * 3
task_num_same_time = 50
slurm_folder = os.path.abspath("./slurm_results")
# slurm_scripts = os.path.join(slurm_folder, 'slurm_scripts')
slurm_output = os.path.join(slurm_folder, job_name)
os.makedirs(slurm_output, exist_ok=True)
sbatch_header = (
    f"#!/bin/bash \n"
    f"#SBATCH --job-name={job_name} \n"
    f"#SBATCH --output={slurm_output}/%x_%A_%a.out \n"
    f"#SBATCH --error={slurm_output}/%x_%A_%a.err \n"
    f"#SBATCH --array=0-{task_num-1}%{task_num_same_time} \n"
    # f"#SBATCH --gres=gpu:1 \n"
    f"#SBATCH --cpus-per-task={threads} \n"
    f"#SBATCH --partition={partition} \n\n"
)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
clair3_model = "clair3/bin/models/r1041_e82_400bps_sup_v520"
with open(f"{os.path.join(slurm_folder, job_name)}.sbatch", "w") as sbatch_file:
    sbatch_file.write(sbatch_header)
    # sbatch_file.write("export CUDA_VISIBLE_DEVICES=$SLURM_ARRAY_TASK_ID\n")
    sbatch_file.write("declare -a commands\n")
    for index, row in manifest.iterrows():
        bed_file_loc = bed_file_dict[row.bed_file]
        input_bam = glob.glob(f"{os.path.join(output_folder, row.sample_id, f"calls*.bam")}")[0]
        read_group_prefix = get_readgroup_prefix_and_full_rg(input_bam)
        rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, "SQK-NBD114-96")
        for index_inner, rg in enumerate(rg_list[:3]):
            medaka_output = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_tr", 'medaka_to_ref.TR.vcf')
            tdb_output  = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_tr", f"{row.sample_id}_{barcode_list[index_inner]}.tdb")
            cmd = (
                f"commands[{index * 3 + index_inner}]=\"tdb create {medaka_output} -o {tdb_output}  "
                f"-s {rg} \"\n"
            )
            sbatch_file.write(cmd)
    sbatch_file.write("\n${commands[$SLURM_ARRAY_TASK_ID]}\n")

In [300]:
from collections import defaultdict
import os

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
output_vcf_folder = 'analysis/sv/kanpig/'

# Group VCFs by bed_file and keep original bed_file names
bedfile_to_vcfs = defaultdict(list)
bedfile_to_original_name = {}

for index, row in manifest.iterrows():
    bed_file = bed_file_dict[row.bed_file]
    bedfile_to_original_name[bed_file] = row.bed_file  # Track original name

    input_sam = os.path.join(output_folder, row.sample_id, f"{row.sample_id}.sam")
    read_group_prefix = get_readgroup_prefix_and_full_rg(input_sam)
    rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, kit_name)

    for index_inner, rg in enumerate(rg_list[:3]):
        sniffles_output_vcf = os.path.join(
            output_folder,
            row.sample_id,
            f"{row.sample_id}_{barcode_list[index_inner]}_global.germline.vcf.gz"
        )
        bedfile_to_vcfs[bed_file].append(sniffles_output_vcf)

# Print bcftools merge command per bed_file
for bed_file, vcf_list in bedfile_to_vcfs.items():
    original_name = bedfile_to_original_name[bed_file]
    output_vcf = os.path.join(output_vcf_folder, f"merged_{original_name}.vcf.gz")
    input_vcfs = " ".join(vcf_list)
    cmd = f"bcftools merge --merge none -R {bed_file} {input_vcfs} -Oz -o {output_vcf}"
    print(cmd)


reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
output_vcf_folder = 'analysis/trio_analysis/snv_trio/'

# Group VCFs by sample_id
sample_to_vcfs = defaultdict(list)

for index, row in manifest.iterrows():
    bed_file = bed_file_dict[row.bed_file]  # still needed for -R option

    input_bam = glob.glob(f"{os.path.join(output_folder, row.sample_id, f"calls*.bam")}")[0]
    read_group_prefix = get_readgroup_prefix_and_full_rg(input_bam)
    rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, kit_name)

    # Expecting first 3 barcodes (proband, mother, father)
    for index_inner, rg in enumerate(rg_list[:3]):
        sniffles_output_vcf = os.path.join(
            output_folder,
            row.sample_id,
            f"{row.sample_id}_{barcode_list[index_inner]}.local.phased.vcf.gz"
        )
        sample_to_vcfs[row.sample_id].append((sniffles_output_vcf, bed_file))

# Print bcftools merge command per sample
for sample_id, vcf_bed_list in sample_to_vcfs.items():
    # Assume all VCFs for a sample share the same bed_file
    bed_file = vcf_bed_list[0][1]
    input_vcfs = " ".join(v[0] for v in vcf_bed_list)
    output_vcf = os.path.join(output_vcf_folder, f"{sample_id}.snv.merged.vcf.gz")
    if bed_file == None:
        cmd = f"bcftools merge {input_vcfs} -W -Oz -o {output_vcf}"
    else:
        cmd = f"bcftools merge -R {bed_file} {input_vcfs} -W -Oz -o {output_vcf}"
    # print(cmd)

reference = "GRCh38-2.1.0_genome_mainchrs.fa"
model = 'sup,5mCG_5hmCG'
kit_name = 'SQK-NBD114-96'
output_folder = 'GREGoR_adaptive_sampling/'
output_vcf_folder = 'analysis/trio_analysis/tr_trio/'

# Group VCFs by sample_id
family_to_samples = defaultdict(list)

for index, row in manifest.iterrows():
    bed_file = bed_file_dict[row.bed_file]  # still needed for -R option

    input_bam = glob.glob(f"{os.path.join(output_folder, row.sample_id, f"calls*.bam")}")[0]
    read_group_prefix = get_readgroup_prefix_and_full_rg(input_bam)
    rg_list, barcode_list = extract_barcodes(row.sample_id, read_group_prefix, kit_name)

    # Expecting first 3 barcodes (proband, mother, father)
    for index_inner, rg in enumerate(rg_list[:3]):
        tdb_output = os.path.join(output_folder, row.sample_id, f"{row.sample_id}_{barcode_list[index_inner]}_tr", f"{row.sample_id}_{barcode_list[index_inner]}.tdb")
        family_to_samples[row.sample_id].append((tdb_output, bed_file))
# Print bcftools merge command per sample

for sample_id, vcf_bed_list in family_to_samples.items():
    # Assume all VCFs for a sample share the same bed_file
    bed_file = vcf_bed_list[0][1]
    input_vcfs = " ".join(v[0] for v in vcf_bed_list)
    output_vcf = os.path.join(output_vcf_folder, f"{sample_id}.tdb")
    cmd = f"tdb merge {input_vcfs} -o {output_vcf}"
    # print(cmd)